# Arabic Cursive OCR Benchmark — Hosted APIs on Colab

Runs Gemini / OpenAI / Claude against the **Muharaf** test set via their APIs.
No GPU needed. Add whichever of these Colab secrets you have keys for:
`GEMINI_API_KEY`, `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`
(a free Gemini key: https://aistudio.google.com/apikey).

## 1. Clone + install

In [ ]:
# Clone the (private) benchmark repo using a fine-grained PAT stored as a
# Colab secret named GH_TOKEN.  See notebooks/README.md to create one.
import os, subprocess
GH_USER, REPO = 'rabeeh97', 'arabic-ocr-benchmark'
token = None
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception as e:
    print('no GH_TOKEN secret:', e)
if not os.path.isdir(REPO):
    auth = f'{token}@' if token else ''
    url = f'https://{auth}github.com/{GH_USER}/{REPO}.git'
    subprocess.run(['git', 'clone', '--depth', '1', url], check=True)
os.chdir(REPO)
print(subprocess.run(['git','log','--oneline','-1'], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -e '.'  # core only — no torch/transformers

In [ ]:
# Keep all downloads on Colab's own scratch disk (ephemeral, ~78 GB) --
# never the local machine. Nothing here touches your computer.
import os
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Optional HF_TOKEN secret: lifts anonymous rate limits and speeds the
# Muharaf download (and is required for gated models like PaliGemma's base).
try:
    from google.colab import userdata
    _hf = userdata.get('HF_TOKEN')
    if _hf:
        os.environ['HF_TOKEN'] = _hf
        print('HF_TOKEN set')
except Exception:
    pass

## 2. Load API keys from Colab secrets

In [ ]:
import os
from google.colab import userdata
KEYMAP = {
    'GEMINI_API_KEY': 'gemini-2.5-flash',
    'OPENAI_API_KEY': 'gpt-4o',
    'ANTHROPIC_API_KEY': 'claude-sonnet',
}
MODELS = []
for env_key, model_name in KEYMAP.items():
    try:
        val = userdata.get(env_key)
    except Exception:
        val = None
    if val:
        os.environ[env_key] = val
        MODELS.append(model_name)
print('will run:', MODELS or 'NOTHING — add at least one API key secret')

## 3. Run + report

In [ ]:
run_cfg = {
    'models': MODELS,
    'datasets': ['muharaf_lines'],
    'limit': 25,            # keep small — paid APIs. bump when happy.
    'prompt_version': 'line-v1',
    'normalization_profiles': ['raw', 'light', 'diac_insensitive'],
    'concurrency': 4,
    'results_dir': 'results',
}
from arabic_ocr_bench.runner import run
from arabic_ocr_bench.report import score_run
run_dir = run(run_cfg)
summary_md = score_run(run_dir, profiles=tuple(run_cfg['normalization_profiles']))
print(open(summary_md).read())

## 4. Inspect predictions

In [ ]:
import json, pandas as pd, pathlib
rows = [json.loads(l) for l in (run_dir / 'predictions.jsonl').read_text().splitlines()]
df = pd.DataFrame(rows)[['model','sample_id','ground_truth','cleaned_text','latency_ms','error']]
pd.set_option('display.max_colwidth', 60)
df.head(20)

## 5. Save results to Google Drive

In [ ]:
# Persist this run's results to Google Drive (they vanish when Colab recycles).
from google.colab import drive
drive.mount('/content/drive')
import shutil, pathlib
dst = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench/results') / run_dir.name
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(run_dir, dst, dirs_exist_ok=True)
print('saved ->', dst)

## Notes

- `cost_usd_total` in the summary is estimated from `configs/models.yaml` pricing.
- Gemini's free tier is rate-limited; the runner retries 429s with backoff.
- Refusals (model declines an image) are logged as `error` and scored as empty output.